In [12]:
# PARTIE 1: Définir une classe de contexte structurée appelée ColourContext
from dataclasses import dataclass
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain.messages import HumanMessage

@dataclass
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_colour: str = "yellow"

# PARTIE 2: Agent sans contexte
model = ChatOllama(
    model="llama3.2:3b", 
    temperature=0
)

# The agent is aware of the schema but lacks tools to read it
agent = create_agent(
    model=model,
    context_schema=ColourContext
)

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)
print("Part 2 Response:", response['messages'][-1].content)

Part 2 Response: I don't have any information about your personal preferences, including your favorite color. This conversation just started, and I don't have the ability to know anything about you unless you choose to share it with me. Would you like to tell me what your favorite color is?


In [13]:
# PARTIE 3: Agent avec contexte
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

agent_with_context = create_agent(
    model=model,
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext
)

response_1 = agent_with_context.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)
print("Part 3 Response (Default):", response_1['messages'][-1].content)

# PARTIE 4: Changement de contexte (Injecting a new context state)
response_2 = agent_with_context.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="green")
)
print("Part 4 Response (Changed Context):", response_2['messages'][-1].content)

/Users/mhand/Career/Master/ms-iibdcc/m1/s1/agentic-ai/tp6/.venv/lib/python3.14/site-packages/pydantic/functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  function=lambda v, h: h(v), schema=original_schema
/Users/mhand/Career/Master/ms-iibdcc/m1/s1/agentic-ai/tp6/.venv/lib/python3.14/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


Part 3 Response (Default): The output from the tool call indicates that your favourite colour is blue.


/Users/mhand/Career/Master/ms-iibdcc/m1/s1/agentic-ai/tp6/.venv/lib/python3.14/site-packages/pydantic/functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  function=lambda v, h: h(v), schema=original_schema
/Users/mhand/Career/Master/ms-iibdcc/m1/s1/agentic-ai/tp6/.venv/lib/python3.14/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


Part 4 Response (Changed Context): The output from the tool call indicates that your favourite colour is green.


In [14]:
# PARTIE 5: Définir un état personnalisé d'agent en héritant de AgentState
from langchain.agents import AgentState
from langgraph.types import Command
from langchain.messages import ToolMessage
from langgraph.checkpoint.memory import InMemorySaver

class CustomState(AgentState):
    favourite_colour: str

# PARTIE 6: Agent qui modifie un état
@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(
        update={
            "favourite_colour": favourite_colour,
            "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]
        }
    )

# PARTIE 6: Agent qui récupère un état
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

# Compile the stateful agent
agent_state = create_agent(
    model=model,
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

# Establish the persistent thread configuration
config = {"configurable": {"thread_id": "1"}}

# Interaction 1: Update the state
response_update = agent_state.invoke(
    {"messages": [HumanMessage(content="My favourite colour is green")]},
    config
)
print("Part 6 Response (State Updated):", response_update['messages'][-1].content)

# Interaction 2: Read from the persistent state
response_read = agent_state.invoke(
    {"messages": [HumanMessage(content="What's my favourite colour?")]},
    config
)
print("Part 6 Response (State Retrieved):", response_read['messages'][-1].content)

Part 6 Response (State Updated): It looks like the tool has successfully updated your favourite colour to green! Would you like to ask another question or explore something else?
Part 6 Response (State Retrieved): Your favourite colour is indeed green! Is there anything else I can help you with?
